In [ ]:
import duckdb
from pathlib import Path

db_path = Path("../data/capstone_data.duckdb")
con = duckdb.connect(str(db_path))

print(f"Connected to {db_path}")

Connected to /home/omkar/ds5500/data/capstone_data.duckdb


In [ ]:
# transactions_path = Path("../data/processed/transactions_clean.csv")

# con.execute("""
#     CREATE OR REPLACE TABLE transactions AS 
#     SELECT * FROM read_csv_auto(?)
# """, [str(transactions_path)])

# count = con.execute("SELECT COUNT(*) FROM transactions").fetchone()[0]
# print(f"Loaded transactions: {count:,} rows")

In [ ]:
transactions_path = Path("../data/processed/transactions_clean_v2.csv")

con.execute("""
    CREATE OR REPLACE TABLE transactions_v2 AS 
    SELECT * FROM read_csv_auto(?)
""", [str(transactions_path)])

count = con.execute("SELECT COUNT(*) FROM transactions_v2").fetchone()[0]
print(f"Loaded transactions: {count:,} rows")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded transactions: 8,234,091 rows


In [ ]:
customers_path = Path("../data/processed/customers_clean.csv")

con.execute("""
    CREATE OR REPLACE TABLE customers AS 
    SELECT * FROM read_csv_auto(?)
""", [str(customers_path)])

count = con.execute("SELECT COUNT(*) FROM customers").fetchone()[0]
print(f"Loaded customers: {count:,} rows")

Loaded customers: 276,838 rows


In [9]:
print("\nTable Structure")
schema = con.execute("DESCRIBE transactions_v2").fetchall()
for col_name, col_type, *_ in schema:
    print(f"  {col_name}: {col_type}")

print("\nTable Structure")
schema = con.execute("DESCRIBE customers").fetchall()
for col_name, col_type, *_ in schema:
    print(f"  {col_name}: {col_type}")


Table Structure
  ATM_Address: VARCHAR
  ATM_City_State: VARCHAR
  Merchant_Category: VARCHAR
  Merchant_Name: VARCHAR
  Transaction_Code: BIGINT
  Transaction_Code_Description: VARCHAR
  Transaction_Type: VARCHAR
  Transaction_Type_Description: VARCHAR
  Transaction_Number: VARCHAR
  Amount_Completed: DOUBLE
  Transaction_Date: DATE
  Time_Local_hhmmss: BIGINT
  Recurring_Trxn: VARCHAR
  CustomerID: VARCHAR

Table Structure
  CustomerID: VARCHAR
  Individual: VARCHAR
  Age: DOUBLE
  Gender: VARCHAR
  NAICSCode: DOUBLE
  OriginalCustomerDate: DATE
  RelationshipYears: DOUBLE
  RelationshipMonths: DOUBLE
  BangorWealth: BOOLEAN
  Payroll: BOOLEAN
  Merchant: BOOLEAN
  TPS: BOOLEAN
  PlingCustomer: BOOLEAN
  ABLECustomer: BOOLEAN
  TimeDepositAccount: VARCHAR
  DepositAccount: VARCHAR
  LoanAccount: VARCHAR
  CreditCardAccount: VARCHAR
  ActiveATMCard: VARCHAR
  ActiveDebitCard: VARCHAR
  NumberActiveDDAs: DOUBLE
  NumberActiveTimeDeposits: DOUBLE
  NumberActiveLoans: DOUBLE
  NumberCre

In [10]:
# Running query to check number of unique customers in each table and mutual cutomers in both tables
result = con.execute("""
                    SELECT 
                    (SELECT COUNT(DISTINCT CustomerID) FROM transactions_v2) AS customers_in_transactions,
                    (SELECT COUNT(DISTINCT CustomerID) FROM customers) AS customers_in_demographics,
                    (SELECT COUNT(DISTINCT t.CustomerID) 
                    FROM transactions_v2 t 
                    INNER JOIN customers c ON t.CustomerID = c.CustomerID) AS customers_in_both
                     """).fetchone()

print(f"Unique customers in transactions: {result[0]:,}")
print(f"Unique customers in demographics: {result[1]:,}")
print(f"Customers in BOTH tables:         {result[2]:,}")

Unique customers in transactions: 128,535
Unique customers in demographics: 276,837
Customers in BOTH tables:         68,857


## Customers table had 276,838 rows so maybe one customerID is duplicated. Need to check that

In [11]:
# Find duplicate CustomerID in customers table
result = con.execute("""
    SELECT CustomerID, count(*) as count
    FROM customers
    GROUP BY CustomerID
    HAVING COUNT(*) > 1
""").fetchone()[0]

print(result)

BSB004527


In [12]:
rows = con.execute("""
    SELECT * FROM customers WHERE CustomerID = ?
""", [result]).df()

print(rows.T)

                                              0                    1
CustomerID                            BSB004527            BSB004527
Individual                                    Y                    Y
Age                                        67.0                 76.0
Gender                                        M                    M
NAICSCode                                   NaN                  NaN
OriginalCustomerDate        1983-01-20 00:00:00  1975-05-02 00:00:00
RelationshipYears                          43.0                 51.0
RelationshipMonths                        516.0                608.0
BangorWealth                              False                False
Payroll                                   False                False
Merchant                                  False                False
TPS                                       False                False
PlingCustomer                             False                False
ABLECustomer                      